In [1]:
import os
import sys
import pprint
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.tools import AgentTool
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import McpWorkbench, StdioServerParams, mcp_server_tools

# Project import paths
# Get the current directory (which is TFM_BA_GPT/test)
notebook_dir = os.getcwd()

# Define base project directory (one level up from test/)
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))

# Server root: one level up from project_root
code_root = os.path.abspath(os.path.join(project_root, '..'))
server_root = os.path.join(code_root, 'mcp_servers')



# Define paths to each server and Python executable
PyMCP_FS_path = os.path.join(server_root, 'PyMCP-FS')
Mermaid_MCP_path = os.path.join(server_root, 'Mermaid-MCP')
PDF_format_path = os.path.join(server_root, 'Final_deliverable_MCP_server')
tmp_dir = os.path.join(server_root, 'tmp')

# Python executables
PyMCP_FS_python = os.path.join(PyMCP_FS_path, '.venv', 'bin', 'python')
Mermaid_MCP_python = os.path.join(Mermaid_MCP_path, 'venv', 'bin', 'python')
PDF_format_python = os.path.join(PDF_format_path, 'venv', 'bin', 'python')




# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

from model_loader import load_model_client

In [ ]:
model_client = load_model_client("azure")

In [3]:
# Helpful sanity prints:
print("raw:", getattr(model_client, "_raw_config", None))
print("create_args:", getattr(model_client, "_create_args", None))

raw: {'parallel_tool_calls': False, 'model': 'gpt-oss-20b', 'api_key': 'LMStudio_key', 'model_info': {'vision': False, 'function_calling': True, 'json_output': True, 'family': 'unknown', 'structured_output': True, 'multiple_system_messages': True}, 'base_url': 'http://127.0.0.1:1234/v1'}
create_args: {'parallel_tool_calls': False, 'model': 'gpt-oss-20b'}


In [4]:
# Server params
PyMCP_FS = StdioServerParams(
    command=PyMCP_FS_python,
    args=[
        os.path.join(PyMCP_FS_path, 'main.py'),
        '-d', project_root,
        "/tmp/",
        '--log-level', 'DEBUG',
        '--log-file', 'mcp_debug.log'
    ]
)

Mermaid_MCP = StdioServerParams(
    command=Mermaid_MCP_python,
    args=[
        os.path.join(Mermaid_MCP_path, 'Mermaid-MCP.py'),
        '-d', project_root,
        '-d', tmp_dir
    ]
)

PDF_format_mcp_server = StdioServerParams(
    command=PDF_format_python,
    args=[
        os.path.join(PDF_format_path, 'PDF_format_mcp_server.py'),
        '-d', os.path.join(project_root, 'experience')
    ]
)



In [5]:
PyMCP_FS_mcp_tools = await mcp_server_tools(PyMCP_FS)
mermaid_mcp_tools = await mcp_server_tools(Mermaid_MCP)
pdf_format_mcp_tools = await mcp_server_tools(PDF_format_mcp_server)

In [6]:

# ---- terminate_agent system message (MOVE + clean + render_mermaid_all + robust PDF) ----
terminate_system_message = r"""
You are the TERMINATE_AGENT. Perform the final export pipeline and then end with **TERMINATE**.

**INPUTS**
  project_root:     /run/media/jun/Jun_6T/Code/TFM_BA_GPT/
  conversation log: project_root/conversation/interview.jsonl
  spec:             project_root/conversation/functional_specification.md
  mermaid:          project_root/conversation/process_diagram.mmd

**GOAL**
  Create: project_root/experience/<SESSION_ID>/
      interview.jsonl
      functional_specification.md
      process_diagram.mmd
      process_diagram.png  
      functional_specification.formatted.md
      deliverable.pdf

**STEPS STRICT ORDER**
  1 Read project_root/conversation/interview.jsonl and extract the first available "session_id".
    If multiple lines: pick the first non-empty "session_id".
    If not found: set SESSION_ID = "sess-unknown".

  2 Create folder project_root/experience/<SESSION_ID>/

  3 MOVE the three source files into that folder using 'move_file':
      project_root/conversation/interview.jsonl             -> project_root/experience/<SESSION_ID>/interview.jsonl
      project_root/conversation/functional_specification.md -> project_root/experience/<SESSION_ID>/functional_specification.md
      project_root/conversation/process_diagram.mmd         -> project_root/experience/<SESSION_ID>/process_diagram.mmd

  4 Clean the Mermaid file **in place** and report issues (severity/code/message/recommendation):
      clean_mermaid(
        input_path="project_root/experience/<SESSION_ID>/process_diagram.mmd",
        output_path="project_root/experience/<SESSION_ID>/process_diagram.mmd",
        overwrite=True
      )

  5 Render Mermaid **SVG + PNG + diagram PDF** in one call:
      render_mermaid_all(
        input_path="project_root/experience/<SESSION_ID>/process_diagram.mmd",
        output_svg="project_root/experience/<SESSION_ID>/process_diagram.svg",
        output_png="project_root/experience/<SESSION_ID>/process_diagram.png",
        output_pdf="project_root/experience/<SESSION_ID>/process_diagram.pdf",
        theme="forest"
      )

  6 Run **format_and_export** ONLY:
      format_and_export(
        session_dir="project_root/experience/<SESSION_ID>",
        md="functional_specification.md",
        diagram="process_diagram.png",
        md_out="functional_specification.formatted.md",
        pdf_out="deliverable.pdf",
        engine="xelatex",
        toc=True, toc_depth=2, margin="1in",
        mainfont="DejaVu Serif", monofont="DejaVu Sans Mono"
      )

  7 Confirm project_root/experience/<SESSION_ID>/deliverable.pdf exists.

  8 Output exactly: **TERMINATE**

**RULES**
  Be explicit about each steps success or failure.
  If a tool call fails, retry up to 1 times with the same arguments.
"""


In [7]:
termination = TextMentionTermination("TERMINATE")

terminate_agent = AssistantAgent(
    name="terminate_agent",
    model_client=model_client,
    system_message=terminate_system_message,
    tools=(PyMCP_FS_mcp_tools + mermaid_mcp_tools + pdf_format_mcp_tools),
    reflect_on_tool_use=False,
)
team = RoundRobinGroupChat(
    [terminate_agent],
    termination_condition=termination,
    )
await Console(team.run_stream(task="Execute the termination pipeline"))

---------- TextMessage (user) ----------
Execute the termination pipeline
---------- ToolCallRequestEvent (terminate_agent) ----------
[FunctionCall(id='673338197', arguments='{"path":"project_root/conversation/interview.jsonl"}', name='read_file')]
---------- ToolCallExecutionEvent (terminate_agent) ----------
[FunctionExecutionResult(content='[{"type": "text", "text": "Error calling tool \'read_file\': [Errno 2] No such file or directory: \'/run/media/jun/Jun_6T/Code/TFM_BA_GPT/test/project_root/conversation/interview.jsonl\'", "annotations": null}]', name='read_file', call_id='673338197', is_error=True)]
---------- ToolCallSummaryMessage (terminate_agent) ----------
[{"type": "text", "text": "Error calling tool 'read_file': [Errno 2] No such file or directory: '/run/media/jun/Jun_6T/Code/TFM_BA_GPT/test/project_root/conversation/interview.jsonl'", "annotations": null}]
---------- ToolCallRequestEvent (terminate_agent) ----------
[FunctionCall(id='910284657', arguments='{"path":"proj

TaskResult(messages=[TextMessage(id='d64b864b-5ee3-4006-a61c-355d8c80e0de', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 21, 16, 35, 36, 53146, tzinfo=datetime.timezone.utc), content='Execute the termination pipeline', type='TextMessage'), ToolCallRequestEvent(id='3aa0c0c3-ceb9-4996-89d9-07de690023d5', source='terminate_agent', models_usage=RequestUsage(prompt_tokens=3538, completion_tokens=70), metadata={}, created_at=datetime.datetime(2025, 8, 21, 16, 35, 37, 783659, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='673338197', arguments='{"path":"project_root/conversation/interview.jsonl"}', name='read_file')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='98e7da1b-d00d-4e4a-8031-0b6181eff2be', source='terminate_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 8, 21, 16, 35, 38, 57174, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='[{"type": "text", "text": "Error calling to